# MoodLens — Improved fine-tune (multi-label + class weighting)

Attacks the three reasons v1 lost to the baseline:
1. **Keeps multi-label data** — no more discarding multi-emotion rows.
2. **Class-weighted loss** — stops the model ignoring rare emotions (disgust, fear).
3. **Trains bert-base AND roberta-base**, early-stopped, then compares.

Scoring uses the **same single-label protocol** as the baseline (argmax on the
same 4,968-row test set), so results compare directly to baseline 0.722 / 0.663.

**Before running:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.
Produces a zip of the best model + a comparison table. ~40–60 min for both models.

## 1. Confirm GPU

In [ ]:
!nvidia-smi

## 2. Install dependencies

In [ ]:
!pip install -q transformers==4.47.1 datasets==3.2.0 scikit-learn==1.6.0 accelerate==1.2.1

## 3. Ekman labels + GoEmotions mapping (mirrors app/core/emotions.py)

In [ ]:
EKMAN = ["joy", "anger", "sadness", "fear", "surprise", "disgust", "neutral"]
EKMAN_LABEL2ID = {e: i for i, e in enumerate(EKMAN)}
EKMAN_ID2LABEL = {i: e for e, i in EKMAN_LABEL2ID.items()}

GOEMOTIONS_TO_EKMAN = {
    "amusement": "joy", "excitement": "joy", "joy": "joy", "love": "joy",
    "desire": "joy", "optimism": "joy", "caring": "joy", "pride": "joy",
    "admiration": "joy", "gratitude": "joy", "relief": "joy", "approval": "joy",
    "anger": "anger", "annoyance": "anger", "disapproval": "anger",
    "sadness": "sadness", "disappointment": "sadness", "embarrassment": "sadness",
    "grief": "sadness", "remorse": "sadness",
    "fear": "fear", "nervousness": "fear",
    "surprise": "surprise", "realization": "surprise", "confusion": "surprise",
    "curiosity": "surprise",
    "disgust": "disgust",
    "neutral": "neutral",
}

## 4. Build datasets
- **Train** = multi-label multi-hot, ALL rows with ≥1 Ekman label (the data v1 threw away).
- **Val / Test** = single-label rows only, as one-hot — identical protocol to the baseline
  so the final numbers compare directly.

In [ ]:
from datasets import load_dataset, Dataset

raw = load_dataset("go_emotions", "simplified")
go_names = raw["train"].features["labels"].feature.names

def multihot(label_ids, single_only):
    buckets = {GOEMOTIONS_TO_EKMAN.get(go_names[i]) for i in label_ids}
    buckets.discard(None)
    if not buckets or (single_only and len(buckets) != 1):
        return None
    vec = [0.0] * len(EKMAN)
    for b in buckets:
        vec[EKMAN_LABEL2ID[b]] = 1.0
    return vec

def build(split, single_only):
    texts, labels = [], []
    for ex in split:
        v = multihot(ex["labels"], single_only)
        if v is None:
            continue
        texts.append(ex["text"]) ; labels.append(v)
    return Dataset.from_dict({"text": texts, "labels": labels})

train_ds = build(raw["train"], single_only=False)
val_ds   = build(raw["validation"], single_only=True)
test_ds  = build(raw["test"], single_only=True)
print(f"train (multi-label): {len(train_ds)}  |  val: {len(val_ds)}  |  test: {len(test_ds)}")

## 5. Class weights
`pos_weight[c] = (#negatives / #positives)` for each emotion, capped at 10 so the rare
classes are up-weighted without wildly over-predicting them.

In [ ]:
import torch

counts = torch.tensor(train_ds["labels"]).sum(dim=0)
N = len(train_ds)
pos_weight = ((N - counts) / counts).clamp(max=10.0)
for e, c, w in zip(EKMAN, counts.tolist(), pos_weight.tolist()):
    print(f"{e:9s} positives={int(c):6d}  pos_weight={w:.2f}")

## 6. Training helper
Custom `Trainer` injects the class-weighted BCE loss. Metrics use argmax (top emotion)
vs the one-hot ground truth — the single-label protocol that matches the baseline.

In [ ]:
import numpy as np
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback)


class WeightedTrainer(Trainer):
    def __init__(self, pos_weight=None, **kw):
        super().__init__(**kw)
        self.pos_weight = pos_weight

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss_fct = nn.BCEWithLogitsLoss(pos_weight=self.pos_weight.to(outputs.logits.device))
        loss = loss_fct(outputs.logits, labels.float())
        return (loss, outputs) if return_outputs else loss


def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=-1)
    trues = np.argmax(p.label_ids, axis=-1)
    return {"accuracy": accuracy_score(trues, preds),
            "macro_f1": f1_score(trues, preds, average="macro", zero_division=0)}


def make_collate(tokenizer):
    def collate(features):
        labels = torch.tensor([f["labels"] for f in features], dtype=torch.float)
        feats = [{k: v for k, v in f.items() if k != "labels"} for f in features]
        batch = tokenizer.pad(feats, padding=True, return_tensors="pt")
        batch["labels"] = labels
        return batch
    return collate


def train_and_eval(base, epochs=6, batch_size=16, lr=2e-5):
    tok = AutoTokenizer.from_pretrained(base)
    tok_fn = lambda b: tok(b["text"], truncation=True, max_length=256)
    tr = train_ds.map(tok_fn, batched=True, remove_columns=["text"])
    va = val_ds.map(tok_fn, batched=True, remove_columns=["text"])
    te = test_ds.map(tok_fn, batched=True, remove_columns=["text"])

    model = AutoModelForSequenceClassification.from_pretrained(
        base, num_labels=len(EKMAN), problem_type="multi_label_classification",
        id2label=EKMAN_ID2LABEL, label2id=EKMAN_LABEL2ID)

    short = base.split("/")[-1]
    args = TrainingArguments(
        output_dir=f"ckpt-{short}", num_train_epochs=epochs,
        per_device_train_batch_size=batch_size, per_device_eval_batch_size=64,
        learning_rate=lr, eval_strategy="epoch", save_strategy="epoch",
        load_best_model_at_end=True, metric_for_best_model="macro_f1",
        greater_is_better=True, save_total_limit=1, logging_steps=100,
        report_to="none")

    trainer = WeightedTrainer(
        pos_weight=pos_weight, model=model, args=args,
        train_dataset=tr, eval_dataset=va, tokenizer=tok,
        data_collator=make_collate(tok), compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])

    trainer.train()
    res = trainer.evaluate(te)
    trainer.save_model(f"ekman-{short}") ; tok.save_pretrained(f"ekman-{short}")
    print(f"\n=== {short} TEST ===  acc={res['eval_accuracy']:.4f}  macro_f1={res['eval_macro_f1']:.4f}\n")
    return {"accuracy": round(res["eval_accuracy"], 4), "macro_f1": round(res["eval_macro_f1"], 4)}

## 7. Train both models

In [ ]:
RESULTS = {"baseline (off-the-shelf)": {"accuracy": 0.722, "macro_f1": 0.663}}
for base in ["bert-base-uncased", "roberta-base"]:
    print(f"\n########## Training {base} ##########\n")
    RESULTS[base] = train_and_eval(base)

## 8. Comparison table

In [ ]:
print(f"{'model':28s} {'accuracy':>10s} {'macro_f1':>10s}")
print("-" * 50)
for name, m in RESULTS.items():
    print(f"{name:28s} {m['accuracy']:>10.4f} {m['macro_f1']:>10.4f}")

trained = {k: v for k, v in RESULTS.items() if k != "baseline (off-the-shelf)"}
best = max(trained, key=lambda k: trained[k]["macro_f1"])
beat = trained[best]["macro_f1"] > RESULTS["baseline (off-the-shelf)"]["macro_f1"]
print(f"\nBest: {best} (macro_f1={trained[best]['macro_f1']}) — "
      f"{'BEATS' if beat else 'does NOT beat'} baseline.")

## 9. Download the best model
Unzip into `backend/models/ekman-best/`, set `MODEL_NAME=models/ekman-best` in `backend/.env`.

In [ ]:
import shutil
short = best.split("/")[-1]
shutil.make_archive(f"ekman-{short}", "zip", f"ekman-{short}")
from google.colab import files
files.download(f"ekman-{short}.zip")